# 以 ollama 執行 llama3

In [2]:
import ollama
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np

In [2]:
from ollama import generate

def generate_response(content):
  prefix = '''你是一個厲害的股市分析師，會用繁體中文回答下列問題，當正面時回答#非常好或#好，負面時回答#不好或#非常不好，其他則是#無關'''
  response = generate(
    model='llama3',
    prompt=f'<PRE> {prefix} <SUF>{content} <MID>',
    options={
      'num_predict': 128,
      'temperature': 1,
      'top_p': 0.9,
      'stop': ['<|eot_id|>'],
      'seed' : 0,
    },
  )
  print(f'input{content}')
  print(f'output{response["response"]}')
  return response['response']

In [6]:
def lm_response(instruction, count):
    text = ''
    try:
        text = generate_response(instruction)
        # print(text)
    except:
        pass
    # print(text)

    if text.find('非常不好') > 0:
        score = -2
    elif text.find('非常好') > 0:
        score = 2
    elif text.find('不好') > 0:
        score = -1
    elif text.find('好') > 0:
        score = 1
    else:
        score = 0

    return score  # 分數

In [7]:
content = '''在積體電路產業的領航帶動下，南科2022年營業額達1兆4,833.74億元，成長35.48％，未來，南科將持續持續布局前瞻產業，導入學研技術，並加速推動5G、AI、物聯網等新興應用，以提升產品附加價值，並積極拓展全球市場，以維持長期穩定成長。'''
lm_response(content, 1)

input在積體電路產業的領航帶動下，南科2022年營業額達1兆4,833.74億元，成長35.48％，未來，南科將持續持續布局前瞻產業，導入學研技術，並加速推動5G、AI、物聯網等新興應用，以提升產品附加價值，並積極拓展全球市場，以維持長期穩定成長。
output#非常好 

南科的業績表現真的很強，35.48% 的增長率實在是非常可觀！這種領航帶動的業績成長，並且願意投資學研技術和新興應用，展示了南科的前瞻性和穩定成長的能力。未來，南科將持續推進5G、AI和物聯網等新興應用的發展，以提高產品附加價值，並拓展全球市場，實在是很有希望的！


2

In [4]:
def calculate_return(signal, threshold):
    # signal = [日期, 評分, 價格]
    hold = 0  # 持有的股數
    balance = 10000000
    init_balance = balance

    for item in signal:
        score = float(item[1])
        price = float(item[2])

        if hold == 0:
            if score == 2:
                hold += int(balance/price)
                balance = 0
            elif score == 1:
                hold += int((balance/2)/price)
                balance = int(balance/2)
        elif hold > 0:
            if score < 0:
                balance += price*hold
                hold = 0
            elif score == 2:
                hold += int(balance/price)
                balance = 0
    
    balance += hold*price
    last_price = float(signal[len(signal) - 1][2])
    bnh = last_price / float(signal[0][2])
    r = balance/init_balance

    return round(r, 5), round(bnh, 5)

In [5]:
# main cell
regenerate_count = 1
prompt_mutate_count = 100

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

# stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
# stock_names = ["台泥", "長榮鋼", "群光", "興富發", "華南金", "統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]
# stock_names = ["統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
stock_ids = ["8027"]
stock_names = ["鈦昇"]

p_mutate = '業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢'
rtn_ = 0

for i in range(prompt_mutate_count):
    # 變異
    # p_mutate_2 = "業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢"

    rtns = []
    bnhs = []
    raw_data = []
    for idx, stock_id in enumerate(stock_ids):
        print(f'變異{i + 1}次\t執行{stock_id} {stock_names[idx]}')
        general_prompt = f'請依據"{p_mutate}"對{stock_names[idx]}的影響，當正面時回答#非常好或#好 ，負面時回答#不好或#非常不好，其他則是#無關'
        folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
        files = os.listdir(folder_path)
        sorted_files = sorted(files)

        # 載入買進價格
        df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
        price_np = df.to_numpy()
        prices = price_np.tolist()  # [時間,價格]

        signal_2 = []

        for price in prices:
            # 檢查日期是否在參考日期之後
            try:
                date_obj = datetime.strptime(price[0], '%Y%m%d')
            except:
                continue

            if not (start_time <= date_obj <= end_time):
                continue
            
            sig = 0
            for index, file_name in enumerate(sorted_files):
                if file_name == price[0]:
                    with open(folder_path + file_name, encoding='utf-8') as f:
                        news = f.read()
                        # print(news)
                        prompt = general_prompt + "\n" + news
                        # gemini的回應，回傳帶有買入訊號的list
                        sig = lm_response(prompt, regenerate_count)
                        # print(sig)
                    break
            
            # print(sig)
            if sig != 0 :
                signal_2.append([price[0], sig, price[1]])

        signal_2_np = np.array(signal_2)
        # 回傳 IRR 以及總額投入
        rtn, bnh = calculate_return(signal_2_np, 0)
        rtns.append(rtn)
        bnhs.append(bnh)
        raw_data.append(signal_2_np.tolist())
        # print("rtn", irr)
        # print("總額投入報酬率", bnh)

    # 輸出結果
    write_folder = "evaluate_output/output"
    count = 1
    while True:
        if os.path.exists(write_folder + str(count)):
            count += 1
            continue
        else:
            with open(write_folder + str(count), "a", encoding="UTF-8") as f:
                content = (f'提示：{prompt}\n'
                           f'開始日期 {start_time.strftime("%Y%m%d")},'
                           f'結束日期 {end_time.strftime("%Y%m%d")}\n')
                for j in range(len(stock_ids)):
                    content += (f'股票代號：{stock_ids[j]}\t'
                                f'依照分析報酬率:{rtns[j]}\t'
                                f'總額投入報酬率:{bnhs[j]}\n')
                content += f'變異{i + 1}次，平均報酬率:{sum(rtns) / len(rtns)}，平均總額投入報酬率:{1.170022}\n)'
                content += f'原始數據{raw_data}'
                f.write(content)
            print("寫入" + write_folder + str(count))
            break

    # 判斷是否更改變異
    # if sum(rtns) / len(rtns) > rtn_:
    #     print("更改變異提示為:", p_mutate_2)
    #     p_mutate = p_mutate_2
    #     rtn_ = sum(rtns) / len(rtns)

# todo testing


NameError: name 'datetime' is not defined